# Spectrogram grid: one clip per class, through the package

**Question.** What does each class look like after the project's own preprocessing and features?

- One real clip per class from **fold 1 test**. Fold 1 test has no dredger, so the dredger clip comes from fold 1 train (marked in its title).
- Each clip goes through the package code: `load_audio` → `make_windows` → `log_mel` (slices S2 and S3), with the settings in `configs/default.yaml`.
- All 12 panels share **one colour scale**, so loudness can be compared across classes.
- One clip is not a class: the grid shows what the input looks like, not what every vessel of a class sounds like.

**Inputs on Kaggle:** the `uniqueship-12-class-5-hours` dataset and the private `acousticwise-src` dataset (the `src/acousticwise` package and `configs/default.yaml`, uploaded with `upload_src.sh`). CPU only; about 15 min to join the split zip.

## 1. Settings and the package

In [ ]:
import glob
import os
import posixpath
import subprocess
import sys
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

LOCAL_ZIP = os.environ.get("GRID_ZIP")  # only for a local smoke test on a tiny fake bundle
SEED = 0

src_hits = glob.glob("/kaggle/input/**/acousticwise/__init__.py", recursive=True)
SRC = Path(src_hits[0]).parent.parent if src_hits else Path("../../src").resolve()
CONFIG = next(iter(glob.glob("/kaggle/input/**/configs/default.yaml", recursive=True)), "../../configs/default.yaml")
sys.path.insert(0, str(SRC))

from acousticwise.audio.features import log_mel
from acousticwise.audio.io import load_audio
from acousticwise.audio.preprocess import make_windows

cfg = yaml.safe_load(open(CONFIG))
A, FE = cfg["audio"], cfg["features"]
print("package:", SRC)
print("audio:", A)
print("features:", FE)

OUT = Path("/kaggle/working" if os.path.isdir("/kaggle/working") else "grid_out")
CLIPS = OUT / "clips"
CLIPS.mkdir(parents=True, exist_ok=True)

## 2. Open the bundle

Same as the spike: join the split zip once on Kaggle's temporary disk, then read only the files we need.

In [ ]:
if LOCAL_ZIP:
    ZIP = LOCAL_ZIP
else:
    ZIP = "/tmp/joined.zip"
    if not os.path.exists(ZIP + ".done"):
        hits = glob.glob("/kaggle/input/**/allclass_5hours.z01", recursive=True)
        assert hits, "Attach the uniqueship-12-class-5-hours dataset (Add Input)."
        D = os.path.dirname(hits[0])
        os.makedirs("/tmp/parts", exist_ok=True)
        for src, dst in [("allclass_5hours.z01", "allclass_5hours.z01"),
                         ("allclass_5hours.zip.part", "allclass_5hours.zip")]:
            if not os.path.lexists(f"/tmp/parts/{dst}"):
                os.symlink(f"{D}/{src}", f"/tmp/parts/{dst}")
        t = time.time()
        subprocess.run("cd /tmp && rm -f joined.zip && zip -q -s 0 parts/allclass_5hours.zip --out joined.zip",
                       shell=True, check=True)
        open(ZIP + ".done", "w").close()
        print(f"joined in {time.time() - t:.0f} s")

zf = zipfile.ZipFile(ZIP)
names = zf.namelist()
csv_name = {s: next(n for n in names if n.endswith(f"fold_1/metadata_5s_{s}.csv")) for s in ("train", "test")}
ROOT = csv_name["test"].rsplit("fold_1/", 1)[0]

## 3. Pick one clip per class

A random full-length (5.0 s) clip per class, with a fixed seed so the grid can be rebuilt. Each chosen clip is extracted to `clips/` and listed in `clips.csv`.

In [ ]:
def read(split):
    df = pd.read_csv(zf.open(csv_name[split]), dtype={"MMSI": str, "date": str}, low_memory=False)
    df["member"] = [posixpath.normpath(f"{ROOT}fold_1/{p}") for p in df.path]
    df["split"] = split
    return df


test, train = read("test"), read("train")
CLASSES = sorted(train.label.unique())
rows = []
for c in CLASSES:
    pool = test[test.label == c]
    if pool.empty:
        pool = train[train.label == c]
    if "duration_sec" in pool and (pool.duration_sec >= A["window_seconds"]).any():
        pool = pool[pool.duration_sec >= A["window_seconds"]]
    rows.append(pool.sample(1, random_state=SEED).iloc[0])
chosen = pd.DataFrame(rows).reset_index(drop=True)

paths = []
for r in chosen.itertuples():
    path = CLIPS / f"{r.label}.wav"
    path.write_bytes(zf.read(r.member))
    paths.append(path)
chosen["clip_id"] = [m.split("/audio/", 1)[1].removesuffix(".wav") for m in chosen.member]
chosen[["label", "split", "clip_id", "MMSI"]].to_csv(OUT / "clips.csv", index=False)
print(chosen[["label", "split", "clip_id", "MMSI"]].to_string())

## 4. Through the package: load → window → log-mel

`load_audio` reads the WAV as mono float32 at the configured rate, `make_windows` pads or cuts it to 5 s windows (a 5 s clip gives exactly one), and `log_mel` gives the 64 × 313 dB picture the CNN sees.

In [ ]:
mels = {}
for r, path in zip(chosen.itertuples(), paths):
    samples = load_audio(path, target_sr=A["sample_rate"])
    windows = make_windows(samples, sr=A["sample_rate"], window_seconds=A["window_seconds"],
                           hop_seconds=A["hop_seconds"], min_seconds=A["min_seconds"])
    mels[r.label] = log_mel(windows[0], sr=A["sample_rate"], n_mels=FE["n_mels"],
                            n_fft=FE["n_fft"], hop_length=FE["hop_length"])
    assert mels[r.label].shape == (FE["n_mels"], 313), mels[r.label].shape
print({c: m.shape for c, m in mels.items()})

## 5. The grid

Time runs left to right (5 s), mel band from low (bottom) to high (top). One colour scale for all panels: the 1st to 99th percentile of every value in the grid.

In [ ]:
import matplotlib.pyplot as plt

all_db = np.concatenate([m.ravel() for m in mels.values()])
vmin, vmax = np.percentile(all_db, [1, 99])
nyquist_khz = A["sample_rate"] / 2000

fig, axes = plt.subplots(3, 4, figsize=(16, 9), sharex=True, sharey=True, constrained_layout=True)
for ax, r in zip(axes.flat, chosen.itertuples()):
    im = ax.imshow(mels[r.label], origin="lower", aspect="auto", cmap="magma", vmin=vmin, vmax=vmax,
                   extent=[0, A["window_seconds"], 0, FE["n_mels"]])
    ax.set_title(str(r.label) + (" (fold 1 train)" if r.split == "train" else ""), fontsize=12)
for ax in axes[-1]:
    ax.set_xlabel("time (s)")
for ax in axes[:, 0]:
    ax.set_ylabel(f"mel band (0 to {nyquist_khz:.0f} kHz)")
fig.colorbar(im, ax=axes, shrink=0.8, label="power (dB)")
fig.suptitle("UniqueShip fold 1: one 5 s clip per class, log-mel as the model sees it", fontsize=14)
plt.savefig(OUT / "spectrogram_grid.png", dpi=150)
plt.show()
print("saved", OUT / "spectrogram_grid.png")